# Multiomics correction evaluation plots

This notebook mirrors the diagnostic evaluation notebooks for the other datasets. It compares before correction and central limma correction for each Quartet omics layer and for the equal-weight all-modality k-means matrix.

In [ ]:
library(tidyverse)

DATA_PATH <- "../evaluation_data/multiomics"
PLOTS_DIR <- "plots/multiomics"
dir.create(PLOTS_DIR, showWarnings = FALSE, recursive = TRUE)

MODALITIES <- c("Transcriptomics", "Proteomics", "Metabolomics")
DONOR_LEVELS <- c("D5", "D6", "F7", "M8")
STATE_LEVELS <- c("Before correction", "After limma correction", "After FedRBE correction")
TOP_PCA_FEATURES <- 3000
MAX_DISTRIBUTION_FEATURES <- 3000
set.seed(1)

condition_palette <- c(
  "D5" = "#44abe7",
  "D6" = "#c55702",
  "F7" = "#009E73",
  "M8" = "#F0E442"
)

## Helpers

The functions below keep this notebook independent of optional plotting packages. Variance explained is estimated with fixed-effect nested linear models, which gives the same kind of condition-vs-batch diagnostic used in the other evaluation notebooks.

In [ ]:
read_feature_matrix <- function(path) {
  df <- readr::read_tsv(path, show_col_types = FALSE)
  first_col <- names(df)[1]
  mat <- df %>%
    column_to_rownames(first_col) %>%
    as.data.frame(check.names = FALSE)
  mat[] <- lapply(mat, as.numeric)
  as.matrix(mat)
}

load_modality_data <- function(modality) {
  before <- read_feature_matrix(file.path(DATA_PATH, "before", modality, "central_intensities_log_UNION.tsv"))
  after <- read_feature_matrix(file.path(DATA_PATH, "after", modality, "intensities_log_Rcorrected_UNION.tsv"))
  metadata <- readr::read_tsv(file.path(DATA_PATH, "before", modality, "metadata.tsv"), show_col_types = FALSE) %>%
    mutate(
      condition = factor(condition, levels = DONOR_LEVELS),
      batch_code = factor(batch_code),
      platform = factor(platform),
      rep = factor(rep)
    )

  missing_before <- setdiff(metadata$file, colnames(before))
  missing_after <- setdiff(metadata$file, colnames(after))
  if (length(missing_before) > 0 || length(missing_after) > 0) {
    stop(modality, ": sample mismatch between metadata and matrices.", call. = FALSE)
  }

  shared_features <- intersect(rownames(before), rownames(after))
  before <- before[shared_features, metadata$file, drop = FALSE]
  after <- after[shared_features, metadata$file, drop = FALSE]

  list(before = before, after = after, metadata = metadata)
}

load_combined_data <- function() {
  before <- read_feature_matrix(file.path(DATA_PATH, "after", "all_modalities_before_kmeans_matrix.tsv"))
  after <- read_feature_matrix(file.path(DATA_PATH, "after", "all_modalities_corrected_kmeans_matrix.tsv"))
  metadata <- readr::read_tsv(file.path(DATA_PATH, "after", "all_modalities_metadata.tsv"), show_col_types = FALSE) %>%
    transmute(
      file = pseudo_sample,
      pseudo_sample = pseudo_sample,
      condition = factor(condition, levels = DONOR_LEVELS),
      batch_code = factor(batch_code),
      rep = factor(rep),
      platform = factor("all_modalities")
    )

  before <- before[, metadata$file, drop = FALSE]
  after <- after[, metadata$file, drop = FALSE]
  list(before = before, after = after, metadata = metadata)
}

prepare_pca_scores <- function(expr, metadata, state_label, top_n = TOP_PCA_FEATURES) {
  feature_var <- apply(expr, 1, var, na.rm = TRUE)
  keep <- which(is.finite(feature_var) & feature_var > 0)
  keep <- keep[order(feature_var[keep], decreasing = TRUE)]
  keep <- head(keep, min(top_n, length(keep)))
  if (length(keep) < 2) stop("Need at least two variable features for PCA.", call. = FALSE)

  pca <- prcomp(t(expr[keep, , drop = FALSE]), center = TRUE, scale. = TRUE)
  variance <- pca$sdev^2 / sum(pca$sdev^2)

  as_tibble(pca$x[, 1:2], rownames = "file") %>%
    left_join(metadata, by = "file") %>%
    mutate(
      state = factor(state_label, levels = STATE_LEVELS),
      pc1_label = paste0("PC1 [", round(variance[1] * 100, 1), "%]"),
      pc2_label = paste0("PC2 [", round(variance[2] * 100, 1), "%]")
    )
}

save_pca_plots <- function(before, after, metadata, label, filename_prefix, shape_col = "platform", after_state_label = "After limma correction") {
  pca_df <- bind_rows(
    prepare_pca_scores(before, metadata, "Before correction"),
    prepare_pca_scores(after, metadata, after_state_label)
  )

  donor_plot <- ggplot(pca_df, aes(PC1, PC2, color = condition, shape = .data[[shape_col]])) +
    geom_point(size = 2.4, alpha = 0.9) +
    facet_wrap(~ state, scales = "free") +
    scale_color_manual(values = condition_palette, drop = FALSE) +
    theme_classic() +
    labs(title = paste(label, "PCA by donor"), x = "PC1", y = "PC2", color = "donor", shape = shape_col)

  batch_plot <- ggplot(pca_df, aes(PC1, PC2, color = batch_code, shape = condition)) +
    geom_point(size = 2.4, alpha = 0.9) +
    facet_wrap(~ state, scales = "free") +
    theme_classic() +
    theme(legend.text = element_text(size = 7), legend.title = element_text(size = 9)) +
    labs(title = paste(label, "PCA by batch"), x = "PC1", y = "PC2", color = "batch", shape = "donor")

  donor_path <- file.path(PLOTS_DIR, paste0(filename_prefix, "_pca_by_donor.png"))
  batch_path <- file.path(PLOTS_DIR, paste0(filename_prefix, "_pca_by_batch.png"))
  ggsave(donor_path, donor_plot, width = 8.5, height = 4.5, bg = "white")
  ggsave(batch_path, batch_plot, width = 9.5, height = 5.2, bg = "white")

  tibble(plot = c("pca_by_donor", "pca_by_batch"), path = c(donor_path, batch_path))
}

sample_distribution_features <- function(before, after, max_features = MAX_DISTRIBUTION_FEATURES) {
  common <- intersect(rownames(before), rownames(after))
  if (length(common) > max_features) sample(common, max_features) else common
}

save_distribution_plot <- function(before, after, metadata, label, filename_prefix, after_state_label = "After limma correction") {
  features <- sample_distribution_features(before, after)
  long_df <- bind_rows(
    before[features, , drop = FALSE] %>%
      as.data.frame(check.names = FALSE) %>%
      rownames_to_column("feature") %>%
      pivot_longer(-feature, names_to = "file", values_to = "value") %>%
      mutate(state = "Before correction"),
    after[features, , drop = FALSE] %>%
      as.data.frame(check.names = FALSE) %>%
      rownames_to_column("feature") %>%
      pivot_longer(-feature, names_to = "file", values_to = "value") %>%
      mutate(state = after_state_label)
  ) %>%
    left_join(metadata, by = "file") %>%
    mutate(state = factor(state, levels = STATE_LEVELS))

  p <- ggplot(long_df, aes(x = batch_code, y = value, fill = batch_code)) +
    geom_violin(scale = "width", trim = TRUE, linewidth = 0.2) +
    facet_wrap(~ state, ncol = 2) +
    theme_minimal() +
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1), legend.position = "none") +
    labs(title = paste(label, "sample-value distributions"), x = "batch", y = "value")

  out_path <- file.path(PLOTS_DIR, paste0(filename_prefix, "_violin_by_batch.png"))
  ggsave(out_path, p, width = 10, height = 5.3, bg = "white")
  tibble(plot = "violin_by_batch", path = out_path)
}

term_variance_table <- function(expr, metadata, state_label) {
  model_data <- metadata %>% mutate(condition = factor(condition), batch_code = factor(batch_code))
  y <- t(expr)
  y_centered <- sweep(y, 2, colMeans(y, na.rm = TRUE), "-")
  sst <- colSums(y_centered^2)
  sst[!is.finite(sst) | sst == 0] <- NA_real_

  full_design <- model.matrix(~ condition + batch_code, data = model_data)
  condition_reduced <- model.matrix(~ batch_code, data = model_data)
  batch_reduced <- model.matrix(~ condition, data = model_data)

  sse <- function(design) {
    residuals <- qr.resid(qr(design), y)
    colSums(residuals^2)
  }

  sse_full <- sse(full_design)
  condition_fraction <- pmax(0, (sse(condition_reduced) - sse_full) / sst)
  batch_fraction <- pmax(0, (sse(batch_reduced) - sse_full) / sst)
  residual_fraction <- pmax(0, sse_full / sst)

  tibble(
    feature = rownames(expr),
    state = factor(state_label, levels = STATE_LEVELS),
    condition = condition_fraction,
    batch = batch_fraction,
    residual = residual_fraction
  ) %>%
    pivot_longer(c(condition, batch, residual), names_to = "term", values_to = "fraction") %>%
    filter(is.finite(fraction)) %>%
    mutate(term = factor(term, levels = c("condition", "batch", "residual")))
}

save_variance_plot <- function(before, after, metadata, label, filename_prefix, after_state_label = "After limma correction") {
  variance_df <- bind_rows(
    term_variance_table(before, metadata, "Before correction"),
    term_variance_table(after, metadata, after_state_label)
  )

  medians <- variance_df %>%
    group_by(state, term) %>%
    summarize(median_fraction = median(fraction, na.rm = TRUE), .groups = "drop")

  p <- ggplot(variance_df, aes(x = term, y = fraction, fill = term)) +
    geom_boxplot(outlier.shape = NA) +
    geom_text(data = medians, aes(y = pmin(median_fraction + 0.04, 0.98), label = sprintf("%.2f", median_fraction)), size = 3) +
    facet_wrap(~ state, ncol = 2) +
    coord_cartesian(ylim = c(0, 1)) +
    theme_minimal() +
    theme(legend.position = "none") +
    labs(title = paste(label, "fixed-effect variance explained"), x = "", y = "fraction of feature variance")

  out_path <- file.path(PLOTS_DIR, paste0(filename_prefix, "_variance_explained.png"))
  table_path <- file.path(PLOTS_DIR, paste0(filename_prefix, "_variance_explained_summary.tsv"))
  ggsave(out_path, p, width = 8.5, height = 4.5, bg = "white")
  write.table(medians, file = table_path, sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

  tibble(plot = "variance_explained", path = out_path)
}

## Per-modality plots

Each modality is evaluated separately because limma correction was performed separately and each omics layer has a different feature scale.

In [ ]:
plot_manifest <- list()
modality_summaries <- list()

for (modality in MODALITIES) {
  message("Plotting ", modality)
  data <- load_modality_data(modality)

  modality_summaries[[modality]] <- tibble(
    dataset = modality,
    features = nrow(data$before),
    samples = ncol(data$before),
    batches = n_distinct(data$metadata$batch_code),
    donors = n_distinct(data$metadata$condition)
  )

  plot_manifest[[paste0(modality, "_pca")]] <- save_pca_plots(
    data$before, data$after, data$metadata,
    label = modality,
    filename_prefix = paste0("multiomics_", modality),
    shape_col = "platform"
  )
  plot_manifest[[paste0(modality, "_distribution")]] <- save_distribution_plot(
    data$before, data$after, data$metadata,
    label = modality,
    filename_prefix = paste0("multiomics_", modality)
  )
  plot_manifest[[paste0(modality, "_variance")]] <- save_variance_plot(
    data$before, data$after, data$metadata,
    label = modality,
    filename_prefix = paste0("multiomics_", modality)
  )
}

bind_rows(modality_summaries)

## Combined all-modality plots

The combined matrix is the equal-weight, row-z-scored matrix prepared for all-three-modality k-means. These plots show whether the combined representation shifts from batch structure to donor structure after limma correction.

In [ ]:
combined <- load_combined_data()

plot_manifest[["combined_pca"]] <- save_pca_plots(
  combined$before, combined$after, combined$metadata,
  label = "All modalities",
  filename_prefix = "multiomics_all_modalities",
  shape_col = "rep"
)
plot_manifest[["combined_distribution"]] <- save_distribution_plot(
  combined$before, combined$after, combined$metadata,
  label = "All modalities",
  filename_prefix = "multiomics_all_modalities"
)
plot_manifest[["combined_variance"]] <- save_variance_plot(
  combined$before, combined$after, combined$metadata,
  label = "All modalities",
  filename_prefix = "multiomics_all_modalities"
)

tibble(
  dataset = "All modalities",
  features = nrow(combined$before),
  samples = ncol(combined$before),
  batches = n_distinct(combined$metadata$batch_code),
  donors = n_distinct(combined$metadata$condition)
)

## K-means ARI plot

This summarizes the central k-means sanity check from the multiomics correction notebook.

In [ ]:
metrics <- readr::read_tsv(file.path(DATA_PATH, "after", "all_modalities_kmeans_metrics.tsv"), show_col_types = FALSE) %>%
  mutate(
    matrix = recode(matrix, before = "Before correction", corrected = "After limma correction"),
    matrix = factor(matrix, levels = STATE_LEVELS),
    target = factor(target, levels = c("donor", "batch"))
  )

ari_plot <- ggplot(metrics, aes(x = matrix, y = ARI, fill = target)) +
  geom_col(position = position_dodge(width = 0.7), width = 0.62) +
  geom_hline(yintercept = 0, linewidth = 0.25) +
  coord_cartesian(ylim = c(min(-0.05, min(metrics$ARI, na.rm = TRUE)), 1.05)) +
  theme_minimal() +
  labs(title = "All-modality k-means ARI", x = "", y = "Adjusted Rand index", fill = "target")

ari_path <- file.path(PLOTS_DIR, "multiomics_all_modalities_kmeans_ari.png")
ggsave(ari_path, ari_plot, width = 6, height = 4, bg = "white")
plot_manifest[["combined_kmeans_ari"]] <- tibble(plot = "kmeans_ari", path = ari_path)

metrics

## FedRBE/FedSim plots

If the FedRBE-equivalent outputs from `04_run_fedrbe.ipynb` are present, generate the same before-vs-FedRBE diagnostics. These use `FedSim_corrected_data.tsv`, which is the local XTX/XTY simulation of the FedRBE app in this environment.

In [ ]:
fedsim_manifest <- list()

for (modality in MODALITIES) {
  fedsim_path <- file.path(DATA_PATH, "after", modality, "FedSim_corrected_data.tsv")
  if (!file.exists(fedsim_path)) {
    message("Skipping FedRBE plots for ", modality, ": ", fedsim_path, " not found")
    next
  }

  data <- load_modality_data(modality)
  fedsim <- read_feature_matrix(fedsim_path)
  fedsim <- fedsim[rownames(data$before), data$metadata$file, drop = FALSE]

  fedsim_manifest[[paste0(modality, "_fedsim_pca")]] <- save_pca_plots(
    data$before, fedsim, data$metadata,
    label = paste(modality, "FedRBE"),
    filename_prefix = paste0("multiomics_", modality, "_fedsim"),
    shape_col = "platform",
    after_state_label = "After FedRBE correction"
  )
  fedsim_manifest[[paste0(modality, "_fedsim_distribution")]] <- save_distribution_plot(
    data$before, fedsim, data$metadata,
    label = paste(modality, "FedRBE"),
    filename_prefix = paste0("multiomics_", modality, "_fedsim"),
    after_state_label = "After FedRBE correction"
  )
  fedsim_manifest[[paste0(modality, "_fedsim_variance")]] <- save_variance_plot(
    data$before, fedsim, data$metadata,
    label = paste(modality, "FedRBE"),
    filename_prefix = paste0("multiomics_", modality, "_fedsim"),
    after_state_label = "After FedRBE correction"
  )
}

combined_fedsim_path <- file.path(DATA_PATH, "after", "all_modalities_fedsim_kmeans_matrix.tsv")
if (file.exists(combined_fedsim_path)) {
  combined <- load_combined_data()
  combined_fedsim <- read_feature_matrix(combined_fedsim_path)
  combined_fedsim <- combined_fedsim[rownames(combined$before), combined$metadata$file, drop = FALSE]

  fedsim_manifest[["combined_fedsim_pca"]] <- save_pca_plots(
    combined$before, combined_fedsim, combined$metadata,
    label = "All modalities FedRBE",
    filename_prefix = "multiomics_all_modalities_fedsim",
    shape_col = "rep",
    after_state_label = "After FedRBE correction"
  )
  fedsim_manifest[["combined_fedsim_distribution"]] <- save_distribution_plot(
    combined$before, combined_fedsim, combined$metadata,
    label = "All modalities FedRBE",
    filename_prefix = "multiomics_all_modalities_fedsim",
    after_state_label = "After FedRBE correction"
  )
  fedsim_manifest[["combined_fedsim_variance"]] <- save_variance_plot(
    combined$before, combined_fedsim, combined$metadata,
    label = "All modalities FedRBE",
    filename_prefix = "multiomics_all_modalities_fedsim",
    after_state_label = "After FedRBE correction"
  )
}

plot_manifest <- c(plot_manifest, fedsim_manifest)
bind_rows(fedsim_manifest, .id = "group")

## Plot manifest

All generated plot paths are saved so the notebook can be rerun non-interactively.

In [ ]:
plot_manifest_df <- bind_rows(plot_manifest, .id = "group")
write.table(plot_manifest_df, file = file.path(PLOTS_DIR, "multiomics_evaluation_plot_manifest.tsv"),
            sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

plot_manifest_df

## Session information

In [ ]:
sessionInfo()